### Processing EMTP-RV Parametric Studio outputs

In [ ]:
import pickle
import numpy as np

In [ ]:
from simulation import Simulations

In [ ]:
from utils import create_white_list

In [ ]:
# Directory path (EDIT):
simulation_path = ''
# Model file name (without extension):
name = 'IEEE39_Wind_v5_23_5_2026'

In [ ]:
# List of machine variable names:
varnames = [
    '/Teta_1_SM1',   # rotor angle
    '/Omega_1_SM1',  # rotor speed
    '/PowerAng_SM1', # power angle
    '/Pe_SM1',       # electrical power
    '/vd_SM1',  # d-axis stator voltage
    '/id_SM1',  # d-axis stator current
    '/Ef_SM1',  # EMF voltage (q-axis)
    '/vq_SM1',  # q-axis stator voltage
    '/iq_SM1',  # q-axis stator current
]
# List of machines that are excluded.
exclude = [8, 10]
# List of bus voltages.
buses = np.arange(start=1, stop=30).tolist()
buses.append(39)

# List of simulation keys identifying type
# and location of the short circuit.
variant = 'V1'      # share of renewables
sc_time = 'T100ms'  # short-circuit duration

In [ ]:
white_list = create_white_list(varnames, exclude, buses)
white_list

In [ ]:
read_me = \
    'This dictionary holds "key & value" pairs for three different\n' \
    'short-circuit types --- three-phase (SC3), two-phase (SC2) and\n' \
    'single-phase (SC1) fault --- applied on main buses of the adapted \n' \
    'IEEE New England 39 bus power system with 18% share of renewables.\n' \
    'Total power production equals 6192 MW, of which 5075.8 MW is from\n' \
    'conventional power plants and 1116.13 MW is from renewables, where\n' \
    '742.43 MW is from the Wind Farm and 373.7 MW from the PV plant.\n' \
    '\n' \
    'Each dictionary key has the form: SCX-BUSY, where X is a number that\n' \
    'identifies the type of short-circuit (3, 2, 1) and Y is a bus index.\n' \
    'To each key is assigned a Pandas DataFrame which holds time-domain\n' \
    'signals from the transient analysis of that particular SC type\n' \
    'and location. Analysis is carried out in EMTP-RV, using Parametric\n' \
    'Studio, with a 40 micro-second time step and a 2 ms output resolution.\n' \
    'Signals from conventional generators are prefixed by the "PowerPlant"\n' \
    'word, those from the Wind Farm have a "DEV2" prefix and those from the\n' \
    'PV plant have a "DEV3" prefix. Bus voltages (a, b, c phases) are\n' \
    'prefixed by the bus name. Fault Ride Through (FRT) signal for the\n' \
    'Wind Farm and PV plant are recorded as well.\n' \
    '\n' \
    'Authors:\n' \
    'Ivica Juric-Grgic, Ivan Krolo, Dino Lovric, Petar Sarajcev\n' \
    'University of Split, FESB, Department of Power Engineering,\n' \
    'R. Boskovica 32, HR-21000 Split, Croatia.\n' \
    'Corresponding e-mail: petar.sarajcev@fesb.hr\n' \
    '\n' \
    'License: CC-BY'

In [ ]:
# Build all simulations.
sims = Simulations(simulation_path, name, white_list)
sims.build_all_simulations()

In [ ]:
# Dictionary holding DataFrames of signals from all simulations.
nsim = sims.get_nb_simu_tot()
print(f'Total no. of simulations: {nsim}')
# Dict keys for simulations in order: 30 SC3 followed by 30 SC2 buses.
sim_keys = ['SC3-' + 'BUS'+str(k) for k in buses]
sim_keys.extend(['SC2-' + 'BUS'+str(k) for k in buses])
if nsim != len(sim_keys):
    raise ValueError()

# Renaming select columns.
name_pairs = {
    # Wind farm signals.
    'FFC_WP2/Wind_Turbine/PMSG_T_rotor': 'WindFarm/PMSG_T_rotor',
    'FFC_WP2/Wind_Turbine/PMSG_w_rotor': 'WindFarm/PMSG_w_rotor',
    'FFC_WP2/Converter_control/Control/Grid_Ctrl/FRT_flag': 'WindFarm/FRT_flag',
    # Solar park signals.
    'WECC_PVPark_1/Converter_control/Control/GridControl_DLL/FRT_flag': 'PVPark/FRT_flag',
}

data = {}
data['README'] = read_me
for key, index in zip(sim_keys, range(nsim)):
    # Export signals to DataFrame.
    sim = sims.get_simulation(index)
    sim_df = sim.to_dataframe()
    sim_df.rename(columns=name_pairs, inplace=True)
    # Assign DataFrame to a simulation key.
    data[key] = sim_df

In [ ]:
# Pickle data to the external file.
file_name = variant + '-' + sc_time + '.pkl'
with open(file=file_name, mode='wb') as fp:
    pickle.dump(data, fp)